# [Step 5 - TextLoader] Plain text into LangChain Documents

**MLCourse - Agentic AI - Module 05: Document Loaders**

> Stage in the capstone: stage 1 INGEST: the capstone reads user-supplied documents with exactly these loaders

## What you'll learn

- the anatomy every loader produces: `Document(page_content=..., metadata=...)`
- how `TextLoader` maps one `.txt` file to exactly one Document
- the three-line inspection ritual: count, metadata, content preview
- a real manipulation: stripping Project Gutenberg boilerplate from *Alice*

---

In [1]:
# =====================================================================
# CELL 1 - SHARED SETUP: imports, track discovery, download-once cache
# =====================================================================
# Every notebook in this track opens with the same plumbing so you can
# focus on the concept instead of paths and downloads.
# ---------------------------------------------------------------------

# --- Standard library -------------------------------------------------
import os                      # small file-system chores (sizes, cleanup)
import re                      # regex for finding chapter markers below
import urllib.request          # polite HTTP fetching for our cache helpers
from pathlib import Path       # object-oriented filesystem paths

# --- Lesson-specific imports ------------------------------------------
from langchain_community.document_loaders import TextLoader   # .txt -> Documents

# ---------------------------------------------------------------------
# TRACK WALKER - find the 03_agentic_ai folder by walking UP from the
# current working directory, so every path works no matter where this
# notebook is opened from inside MLCourse.
# ---------------------------------------------------------------------
def _find_track(start: Path) -> Path:
    """Return the absolute path of the 03_agentic_ai track root."""
    for candidate in (start, *start.parents):      # cwd first, then ancestors
        hit = candidate / "03_agentic_ai"
        if hit.is_dir():                           # anchor found
            return hit.resolve()
    raise FileNotFoundError(
        f"No directory named 03_agentic_ai found above {start} - "
        "run this notebook from somewhere inside the MLCourse repo."
    )

TRACK = _find_track(Path.cwd())     # .../MLCourse/03_agentic_ai
DATA = TRACK / "data"               # every dataset of this track lives here
DATA.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# DOWNLOAD-ONCE HELPERS - fetch a remote resource exactly once, then
# always read the local cache. Public servers deserve politeness.
# ---------------------------------------------------------------------
def get_bytes(fname: str, url: str) -> bytes:
    """Return the file's bytes, downloading only on the very first call."""
    target = DATA / fname                        # local cache location
    if target.exists() and target.stat().st_size > 0:
        payload = target.read_bytes()            # cache hit: zero network use
        print(f"[cache] {fname}: {len(payload):,} bytes")
        return payload
    print(f"[fetch] {url}")
    request = urllib.request.Request(url, headers={"User-Agent": "MLCourse/1.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        payload = response.read()                # one blocking download
    target.write_bytes(payload)                  # persist for future runs
    print(f"[saved] {fname}: {len(payload):,} bytes")
    return payload

def get_text(fname: str, url: str, encoding: str = "utf-8-sig") -> str:
    """get_bytes + decode; 'utf-8-sig' quietly drops a BOM if one exists."""
    return get_bytes(fname, url).decode(encoding, errors="replace")

def to_ascii(text: str) -> str:
    """Console-safe view of book text: non-ASCII glyphs become '?'."""
    return text.encode("ascii", errors="replace").decode("ascii")

# ---------------------------------------------------------------------
# Matplotlib inline guard: get_ipython() exists ONLY inside Jupyter,
# hence try/except so the notebook also runs as a plain script.
# ---------------------------------------------------------------------
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

C:\Users\Thoyajaksha Kashyap\AppData\Local\Temp\ipykernel_72604\3025091758.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader   # .txt -> Documents


## 1. Why a loader at all?

Models and vector stores cannot read files. They read TEXT plus LABELS.
LangChain standardises every input format into one tiny shape:

```python
Document(page_content="<the raw text>", metadata={"source": "<where it came from>"})
```

`TextLoader` is the simplest possible mapping:

- **page_content** = the entire file contents, decoded to a string
- **metadata** = just `{"source": "<absolute path>"}`
- **granularity** = ONE Document per FILE

That last point matters more than it looks: because the whole book arrives
as a single Document, deciding smaller units (chapters? paragraphs?) is NOT
the loader's job - it is chunking's job, which is exactly module 06.

**Pro-tip:** treat every loader with the same suspicion. Load, then inspect
count + metadata + preview BEFORE building anything on top.

## 2. Get the material and load it

Our specimen is the full text of *Alice's Adventures in Wonderland* from
Project Gutenberg (~150 KB). First run downloads it into
`data/alice.txt`; every later run reads the cache instantly.

In [2]:
ALICE_URL = "https://www.gutenberg.org/files/11/11-0.txt"

# get_text both fetches AND caches the bytes under DATA/alice.txt,
# then decodes them so we can peek before involving any loader.
raw_text = get_text("alice.txt", ALICE_URL)

# The actual lesson: TextLoader reads FROM DISK and returns Documents.
loader = TextLoader(str(DATA / "alice.txt"), encoding="utf-8")
docs = loader.load()

print(f"number of Documents : {len(docs)}")          # expect 1 - one per file
doc = docs[0]
print(f"metadata           : {doc.metadata}")
print(f"page_content chars : {len(doc.page_content):,}")

print("\n--- first 300 characters of page_content ---")
print(to_ascii(doc.page_content[:300]))

[cache] alice.txt: 151,191 bytes
number of Documents : 1
metadata           : {'source': 'D:\\projects\\python\\MLCourse\\03_agentic_ai\\data\\alice.txt'}
page_content chars : 144,696

--- first 300 characters of page_content ---
*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice?s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    T


## 3. Inspect like you mean it

Notice what the ritual told us:

- `len(docs) == 1` - granularity confirmed: whole file, single Document;
- `metadata["source"]` - an absolute path string, useful later for filters
  and citations;
- the preview shows Project Gutenberg's legal header sitting INSIDE our
  content - boilerplate that would pollute embeddings if we left it in.

Next we quantify the file and locate its internal structure.

In [3]:
# --- Simple stats straight off the loaded Document --------------------
content = doc.page_content
n_chars = len(content)
n_words = len(content.split())
print(f"chars : {n_chars:,}")
print(f"words : {n_words:,}")

# --- Chapter markers: does the raw text carry structure? --------------
# Alice marks chapters with lines like 'CHAPTER I.' / 'CHAPTER XII.'
# This matters for module 06: separators we choose later will interact
# with this structure, so verify it EXISTS before relying on it.
chapter_lines = [
    line.strip()
    for line in content.splitlines()
    if re.match(r"^CHAPTER\s+[IVXLC]+\.?\s*$", line.strip())
]
print(f"\nchapter markers found: {len(chapter_lines)}")
print("first three:", chapter_lines[:3])

chars : 144,696
words : 26,543

chapter markers found: 12
first three: ['CHAPTER I.', 'CHAPTER II.', 'CHAPTER III.']


## 4. One meaningful manipulation: strip the boilerplate

Gutenberg files wrap the actual novel between two marker lines:

- `*** START OF THE PROJECT GUTENBERG EBOOK ...`
- `*** END OF THE PROJECT GUTENBERG EBOOK ...`

Everything outside those markers is licence text - noise for embeddings.
A loader gives you the file verbatim; CLEANING is your move, and here we
make it explicitly on `page_content`.

In [4]:
START_TAG = "*** START OF THE PROJECT GUTENBERG EBOOK"
END_TAG = "*** END OF THE PROJECT GUTENBERG EBOOK"

body = content.split(START_TAG, 1)[1]     # drop everything before START
body = body.split(END_TAG, 1)[0]          # drop everything after END
body = body.strip()

reduction = 100.0 * (n_chars - len(body)) / n_chars
print(f"before : {n_chars:>8,} chars")
print(f"after  : {len(body):>8,} chars  ({reduction:.1f}% removed)")
print(f"words  : {n_words:,} -> {len(body.split()):,}")

print("\n--- cleaned opening (what downstream stages will actually see) ---")
print(to_ascii(body[:280]))

# Build a fresh Document carrying the cleaned payload forward, keeping the
# original source label plus a note about what we did. Metadata is FREE
# space for provenance - use it.
clean_doc = type(doc)(
    page_content=body,
    metadata={**doc.metadata, "cleaned": "gutenberg_boilerplate_stripped"},
)
print("\ncleaned Document metadata:", clean_doc.metadata)

before :  144,696 chars
after  :  144,607 chars  (0.1% removed)
words  : 26,543 -> 26,527

--- cleaned opening (what downstream stages will actually see) ---
11 ***

[Illustration]




Alice?s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a 

cleaned Document metadata: {'source': 'D:\\projects\\python\\MLCourse\\03_agentic_ai\\data\\alice.txt', 'cleaned': 'gutenberg_boilerplate_stripped'}


## 5. Pitfalls worth remembering

**Pitfall - encoding is not optional**: `TextLoader(encoding="utf-8")` raises
on latin-1 bytes (common in older exports). For mixed-quality corpora pass
`autodetect_encoding=True` and let it sniff instead of crashing mid-batch.

**Pro-tip - utf-8 vs utf-8-sig**: some Windows tools prefix files with a BOM
character (`\ufeff`). Decoding with `"utf-8-sig"` drops it automatically -
that is why the course helper `get_text` uses it by default.

**Pitfall - one giant Document is not chunking**: keeping the whole book as
one Document is fine ONLY until embedding time. The loader stage should stay
faithful to the file; size discipline belongs to module 06.

## Takeaway

**`TextLoader` = whole file in, one `Document` out, `source` in metadata.
Loading is faithful, not smart - inspection and cleaning are YOUR moves,
and they cost three print statements.**

## Summary

- Every loader emits `Document(page_content=..., metadata=...)`; TextLoader
  is the minimal case: 1 file -> 1 Document, `{"source": path}`.
- The inspection ritual (count / metadata / preview) catches most ingestion
  surprises immediately - boilerplate, wrong encodings, empty loads.
- Real corpora need explicit cleaning steps (here: Gutenberg markers);
  record what you did in metadata for provenance.
- Structure checks (12 chapter markers present) tell you whether later
  separator choices in module 06 will actually have something to bite on.